In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 21:01:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import pandas as pd
CURATED_DIR = "../data/curated/curated_transactions/part_3.parquet"

# Load the ETL curated transaction level dataset 
curated_df = pd.read_parquet(CURATED_DIR)
print(curated_df.columns.tolist())
print(curated_df.info())

['user_id', 'merchant_abn', 'dollar_value', 'order_id', 'order_datetime', 'name_consumer', 'address', 'state', 'postcode', 'gender', 'consumer_id', 'name_merchant', 'tags', 'consumer_fraud_probability', 'merchant_fraud_probability', 'consumer_is_fraud', 'merchant_is_fraud', 'POA_CODE_2021', 'SA2_CODE_2021', 'mb_count', 'poa_total', 'ratio', 'SA2_NAME_2021', 'disadvantage_score', 'disadvantage_decile', 'adv_disadv_score', 'adv_disadv_decile', 'economic_resources_score', 'economic_resources_decile', 'education_occupation_score', 'education_occupation_decile', 'usual_resident_population', 'SA2_NAME_2021_income', 'IRSAD_score', 'IRSAD_decile', 'median_personal_income_weekly', 'median_family_income_weekly', 'median_household_income_weekly', 'disadvantage_score_is_approximated', 'disadvantage_decile_is_approximated', 'adv_disadv_score_is_approximated', 'adv_disadv_decile_is_approximated', 'economic_resources_score_is_approximated', 'economic_resources_decile_is_approximated', 'education_occu

In [4]:
curated_df = curated_df[[
    "merchant_abn", "user_id", "order_id", "order_datetime", "dollar_value",
    "name_merchant", "tags",
    "consumer_is_fraud", "merchant_is_fraud",
    "consumer_fraud_probability", "merchant_fraud_probability",
    "postcode", "state", "SA2_CODE_2021",
    "adv_disadv_decile", "economic_resources_decile",
]]

In [ ]:
# Grouping trasaction level data by merchants 
df_by_merch = curated_df.groupby("merchant_abn")

# FEATURES PRODUCED: total_revenue, n_transactions, avg_order_value, n_customers
feat_df = pd.DataFrame({
    "total_revenue": df_by_merch["dollar_value"].sum(),          # revenue: merchant's gross dollar sales
    "n_transactions": df_by_merch["order_id"].size(),            # sales: number of orders
    "avg_order_value": df_by_merch["dollar_value"].mean(),       # average order size
    "n_customers": df_by_merch["user_id"].nunique(),             # customer base
})

# FEATURE PRODUCED: orders_per_customer (how often the average customer buys)
feat_df ["orders_per_customer"] = feat_df ["n_transactions"] / feat_df ["n_customers"]

# FEATURE PRODUCED: avg_weekly_transactions (orders per week over the merchant's own trading span)
span_days = (df_by_merch["order_datetime"].max() - df_by_merch["order_datetime"].min()).dt.days + 1
feat_df ["avg_weekly_transactions"] = feat_df["n_transactions"] / (span_days / 7).clip(lower=1)

# FEATURES PRODUCED: repeat_rate (returning customers), top_customer_share (dependence on one customer)
cust = df_by_merch.groupby(["merchant_abn", "user_id"]).agg(n=("order_id", "size"), spend=("dollar_value", "sum"))
cust["is_repeat"] = cust["n"] >= 2
c = cust.groupby(level=0).agg(repeat_rate=("is_repeat", "mean"),
                              top_spend=("spend", "max"), tot_spend=("spend", "sum"))
feat_df["repeat_rate"] = c["repeat_rate"]
feat_df["top_customer_share"] = c["top_spend"] / c["tot_spend"]

# FEATURES PRODUCED: revenue_growth (second half vs first half, -1..+1),
#                    revenue_cv (monthly volatility, lower = steadier), active_months
month = d["order_datetime"].dt.to_period("M")
m = d.groupby(["merchant_abn", month])["dollar_value"].sum().unstack(fill_value=0)
half = m.shape[1] // 2
h1, h2 = m.iloc[:, :half].sum(axis=1), m.iloc[:, half:].sum(axis=1)
feat_df["revenue_growth"] = (h2 - h1) / (h1 + h2)
feat_df["revenue_cv"] = m.std(axis=1) / m.mean(axis=1)
feat_df["active_months"] = (m > 0).sum(axis=1)

AttributeError: 'DataFrameGroupBy' object has no attribute 'groupby'

In [ ]:
# fraud flags are booleans; fraud_value is the dollar value of fraudulent-consumer transactions
fraud_df = curated_df[["merchant_abn", "dollar_value", "consumer_is_fraud", "merchant_is_fraud",
       "consumer_fraud_probability", "merchant_fraud_probability"]].copy()
fraud_df["fraud_value"] = fraud_df["dollar_value"] * fraud_df["consumer_is_fraud"].astype(float)
fraud_by_merch = fraud_df.groupby("merchant_abn")

# FEATURES PRODUCED: consumer_fraud_rate, fraud_value_share, merchant_fraud_rate,
#                    avg_consumer_fraud_prob, max_merchant_fraud_prob
fraud_feat = pd.DataFrame({
    "consumer_fraud_rate": fraud_by_merch ["consumer_is_fraud"].mean(),                    # share of orders from fraudulent consumers
    "fraud_value_share": fraud_by_merch ["fraud_value"].sum() / fraud_by_merch ["dollar_value"].sum(),  # share of dollars from fraud (main risk feature)
    "merchant_fraud_rate": fraud_by_merch ["merchant_is_fraud"].mean(),                    # share of orders flagged as merchant fraud
    "avg_consumer_fraud_prob": fraud_by_merch ["consumer_fraud_probability"].mean(),       # reference only
    "max_merchant_fraud_prob": fraud_by_merch ["merchant_fraud_probability"].max(),        # reference only
})
# no probability recorded = no flagged days, so treat as 0 (not missing)
fraud_feat = fraud_feat.fillna({"avg_consumer_fraud_prob": 0, "max_merchant_fraud_prob": 0})

In [ ]:
# FEATURES PRODUCED: n_postcodes, n_states (geographic spread)
geo_feat = pd.DataFrame({
    "n_postcodes": d.groupby("merchant_abn")["postcode"].nunique(),
    "n_states": d.groupby("merchant_abn")["state"].nunique(),
})

# FEATURE PRODUCED: postcode_hhi (Herfindahl index of spend by postcode; 1 = all spend from one postcode,
#                   so LOWER = more spread out)
pc = curated_df.groupby(["merchant_abn", "postcode"])["dollar_value"].sum()
pc_share = pc / pc.groupby(level=0).transform("sum")
geo_feat["postcode_hhi"] = (pc_share ** 2).groupby(level=0).sum()

# FEATURE PRODUCED: pct_txn_no_sa2 (data-quality caveat: share of a merchant's orders with no SA2/SEIFA match)
geo_feat["pct_txn_no_sa2"] = d["SA2_CODE_2021"].isna().groupby(key).mean()

# FEATURES PRODUCED: w_adv_disadv_decile, w_economic_resources_decile
# (dollar-weighted SEIFA decile of the customers' areas; orders with no SEIFA value are left out, not counted as 0)
for col in ["adv_disadv_decile", "economic_resources_decile"]:
    num = (d[col] * d["dollar_value"]).groupby(key).sum()
    den = d["dollar_value"].where(d[col].notna()).groupby(key).sum()
    geo_feat["w_" + col] = num / den

In [ ]:
q = feat[["n_customers", "avg_order_value", "fraud_value_share", "active_months"]].quantile([0.1, 0.25, 0.75])

# FLAGS PRODUCED: flag_small_base_big_spender, flag_large_base_high_fraud, flag_new_merchant
feat["flag_small_base_big_spender"] = ((feat["n_customers"] <= q.loc[0.25, "n_customers"])
                                       & (feat["avg_order_value"] >= q.loc[0.75, "avg_order_value"]))
feat["flag_large_base_high_fraud"] = ((feat["n_customers"] >= q.loc[0.75, "n_customers"])
                                      & (feat["fraud_value_share"] >= q.loc[0.75, "fraud_value_share"]))
feat["flag_new_merchant"] = feat["active_months"] <= q.loc[0.1, "active_months"]

flag_cols = [c for c in feat if c.startswith("flag_")]
print(feat[flag_cols].sum())
display(feat[feat["flag_large_base_high_fraud"]].sort_values("fraud_value_share", ascending=False).head(10)[
    ["name_merchant", "category", "n_customers", "fraud_value_share", "total_revenue"]])